# E7 results

Compare the 241k retrieval pilot, the 1M-row retrieval run, the SciFact
encoding smoke, and the duplicate control. Change the result paths below to
inspect a different run.


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display
from matplotlib.ticker import MaxNLocator, NullLocator

PILOT_RESULT_PATH = Path("results/E7/retrieval8_cached_pilot/retrieval_cache.json")
SCALE_RESULT_PATH = Path("results/E7/retrieval8_repeated_1m/retrieval_cache_repeated.json")
ENCODING_RESULT_PATH = Path("results/E7/e7_smoke_qscht8pb/encoding.json")
CONTROL_RESULT_DIR = Path("results/E7/retrieval8_duplicate_control")


def read_result(path):
    """Read one run result from the repo root or the notebook directory."""
    path = path.expanduser()
    if not path.is_absolute() and Path.cwd().name == "notebooks":
        path = Path("..") / path
    if not path.is_file():
        print(f"Result not available yet: {path}")
        return None
    with path.open(encoding="utf-8") as file:
        return json.load(file)


def show_table(title, data, columns):
    """Display one metric group with short column names."""
    display(Markdown(f"**{title}**"))
    display(data[list(columns)].rename(columns=columns).round(3))


def with_paper_values(ours, paper, decimals):
    """Format each cell as ours (paper), with a dash for an unavailable value."""
    ours, paper = ours.align(paper)
    measured = ours.map(lambda value: f"{value:.{decimals}f}" if pd.notna(value) else "—")
    reference = paper.map(lambda value: f"{value:.3f}" if pd.notna(value) else "—")
    return measured + " (" + reference + ")"


# CSRv2 paper, Appendix G.1, Tables 15 (p. 28) and 16 (p. 29).
paper_encoding = pd.DataFrame(
    {"Seconds": [159854.091, 159876.478, 159873.263]},
    index=pd.Index(["MRL", "CSR", "CSRv2"], name="Method"),
)
paper_latency = pd.DataFrame(
    [
        [1.402, 1.428, 1.571, 1.748, 3.972, 68.522],
        [0.227, 0.370, 0.633, 0.797, 3.217, 45.722],
    ],
    index=pd.Index(["MRL", "CSRv2"], name="Method"),
    columns=[2, 4, 8, 16, 64, 4096],
)


## Retrieval · 241k cached corpus

The search timer includes scoring and the final top-10. Amortized ms/query
divides mean batch time by the number of queries in a batch.
Both latency plots share the same y-axis limits. Storage is measured per
document, so repeating the corpus has little effect on that panel.


In [ ]:
def show_retrieval(search_result, latency_limit):
    """Display latency, storage and VRAM for one retrieval run."""
    if search_result is None:
        return
    retrieval = pd.json_normalize(search_result["measurements"])
    run = search_result["provenance"]["config"]["run_name"]
    docs, queries = retrieval.iloc[0][["documents", "queries"]]
    distinct = search_result.get("distinct_document_texts")
    distinct_note = f" · {distinct:,} distinct document texts" if distinct is not None else ""
    display(Markdown(f"**{run}** · {docs:,} rows · {queries:,} queries{distinct_note}"))
    retrieval["amortized_ms_per_query"] = (
        retrieval["ms_per_batch_mean"] / retrieval["query_batch_size"]
    )
    retrieval["corpus_bytes_per_doc"] = retrieval["corpus_bytes"] / retrieval["documents"]


    styles = {
        "qwen_dense": ("Qwen3 (full)", "#737373", None),
        "mrl": ("MRL", "#46263b", "o"),
        "csrv2": ("CSRv2", "#c64b32", "D"),
    }
    widths = sorted(retrieval["k"].dropna().unique())
    retrieval["method_name"] = retrieval["method"].map(lambda method: styles.get(method, (method,))[0])
    sweep = retrieval[retrieval["method"].isin(["mrl", "csrv2"])].dropna(subset=["k"])
    if not sweep.empty:
        latency = sweep.pivot(index="method_name", columns="k", values="amortized_ms_per_query")
        latency = latency.reindex([method for method in ("MRL", "CSRv2") if method in latency.index])
        latency.columns = latency.columns.astype(int)
        latency = latency.rename_axis(index="Method", columns=None)
        display(Markdown(
            "**Retrieval latency (ms/query) · Paper Table 16**  \n"
            "Values are ours (paper). A dash indicates an unavailable measurement."
        ))
        display(with_paper_values(latency, paper_latency, decimals=5))

    retrieval["width_label"] = retrieval["k"].astype("Int64").astype("string").fillna("Full")
    methods = {"method_name": "Method", "width_label": "Width"}
    sizes = retrieval[["documents", "queries", "query_batch_size"]].drop_duplicates()
    if len(sizes) == 1:
        documents, queries, batch_size = sizes.iloc[0]
        print(f"{documents:,} documents · {queries:,} queries · batch size {batch_size}")
    show_table("Search latency and throughput", retrieval, {
        **methods, "ms_per_batch_mean": "Mean ms/batch", "ms_per_batch_p95": "p95 ms/batch",
        "amortized_ms_per_query": "Amortized ms/query", "queries_per_second": "Queries/s",
    })
    retrieval["corpus_kib"] = retrieval["corpus_bytes"] / 2**10
    show_table("Corpus storage", retrieval, {
        **methods, "corpus_bytes_per_doc": "Bytes/doc", "corpus_kib": "Corpus KiB",
    })
    memory_fields = {"cuda_memory.peak_allocated_bytes", "cuda_memory.extra_peak_bytes"}
    if memory_fields.issubset(retrieval.columns):
        memory = retrieval.dropna(subset=sorted(memory_fields)).copy()
        if not memory.empty:
            memory["peak_mib"] = memory["cuda_memory.peak_allocated_bytes"] / 2**20
            memory["extra_kib"] = memory["cuda_memory.extra_peak_bytes"] / 2**10
            show_table("Search VRAM", memory, {
                **methods, "peak_mib": "Peak MiB", "extra_kib": "Additional peak KiB",
            })

    figure, axes = plt.subplots(1, 2, figsize=(6.3, 2.5), dpi=150, layout="constrained")
    for axis, metric, label in zip(
        axes,
        ("ms_per_batch_mean", "corpus_bytes_per_doc"),
        ("Mean latency (ms/batch)", "Corpus bytes / document"),
    ):
        for method, rows in retrieval.groupby("method", sort=False):
            name, color, marker = styles.get(method, (method, None, "o"))
            sized = rows.dropna(subset=["k"]).sort_values("k")
            if sized.empty:
                axis.axhline(rows[metric].iloc[0], ls="--", lw=0.9, color=color, label=name)
            else:
                axis.plot(
                    sized["k"], sized[metric], color=color, label=name, lw=1.15,
                    marker=marker, ms=3.5, mfc="white", mew=0.9,
                )
        if widths:
            axis.set_xscale("log", base=2)
            axis.set_xticks(widths, labels=[str(int(width)) for width in widths])
        else:
            axis.set_xticks([])
        axis.xaxis.set_minor_locator(NullLocator())
        axis.set_xlabel("MRL width / CSRv2 $k$", fontsize=8)
        axis.set_ylabel(label, fontsize=8)
        axis.margins(y=0.10)
        axis.tick_params(axis="both", labelsize=8, direction="in", length=3)
        axis.set_axisbelow(True)
        axis.grid(axis="y", color="0.8", linestyle=":", linewidth=0.6)
        for spine in axis.spines.values():
            spine.set_linewidth(0.65)

    axes[0].set_ylim(0, latency_limit)
    axes[0].yaxis.set_major_locator(MaxNLocator(nbins=5))
    axes[0].ticklabel_format(axis="y", style="plain", useOffset=False)
    axes[1].set_yscale("log")
    handles, labels = axes[0].get_legend_handles_labels()
    figure.legend(handles, labels, loc="outside upper center", ncol=len(labels), frameon=False, fontsize=8)
    plt.show()


pilot = read_result(PILOT_RESULT_PATH)
scale = read_result(SCALE_RESULT_PATH)
latency_limit = 1.1 * max(
    (row["ms_per_batch_mean"] for run in (pilot, scale) if run is not None
     for row in run["measurements"]),
    default=1.0,
)
show_retrieval(pilot, latency_limit)

## Retrieval · 1M repeated rows

Job 29197139 repeats and shuffles the cached document rows to reach 1M.
The search uses the original queries and all 11 configurations.


In [ ]:
show_retrieval(scale, latency_limit)

## Encoding · SciFact smoke

Job 29104915 timed encoding for 24 documents and 8 queries.

Wall time covers the encoding pass. The phase chart shows mean milliseconds
per profiled batch on the CUDA stream. The two panels use independent y-axes.
The backbone interval can include waits while the CPU prepares text.

In [ ]:
encoding_result = read_result(ENCODING_RESULT_PATH)
encoding_measurements = encoding_result["measurements"] if encoding_result else None

if encoding_measurements is not None:
    run = encoding_result["provenance"]["config"]["run_name"]
    docs = encoding_measurements["dense"]["document"]["texts"]
    queries = encoding_measurements["dense"]["query"]["texts"]
    display(Markdown(f"**{run}** · {docs:,} documents · {queries:,} queries"))
    encoding = pd.DataFrame([
        {"model": model, "role": role, **stats}
        for model, roles in encoding_measurements.items()
        for role, stats in roles.items()
    ])
    names = {"dense": "Qwen3", "csrv2": "CSRv2"}
    colors = {"Qwen3": "#737373", "CSRv2": "#c64b32"}
    encoding["model"] = encoding["model"].replace(names)
    corpus_encoding = encoding[encoding["role"] == "document"]
    seconds = (
        corpus_encoding.set_index("model")[["seconds"]]
        .rename(index={"Qwen3": "MRL"}, columns={"seconds": "Seconds"})
        .rename_axis("Method")
    )
    display(Markdown(
        "**Corpus encoding time (s) · Paper Table 15**  \n"
        f"Values are ours (paper). Our run timed {docs:,} documents and the paper timed 1M. "
        "Qwen3 is the encoder used for our MRL baseline."
    ))
    display(
        with_paper_values(seconds, paper_encoding, decimals=3)
        .reindex(paper_encoding.index)
        .rename(index={"MRL": "MRL (Qwen3 encoder)"})
    )
    identity = {"model": "Model", "role": "Input"}
    show_table("Encoding totals", encoding, {
        **identity, "texts": "Texts", "seconds": "Seconds", "texts_per_second": "Texts/s",
    })

    phases = {
        "backbone_stream_ms_per_batch_mean": "Backbone",
        "sae_stream_ms_per_batch_mean": "SAE + top-k",
    }
    phases = {key: label for key, label in phases.items()
              if key in encoding and encoding[key].notna().any()}
    if phases:
        show_table("Encoding breakdown", encoding, {
            **identity, "profiled_batches": "Profiled batches",
            **{key: f"{label} ms/batch" for key, label in phases.items()},
        })
        figure, axes = plt.subplots(
            1, len(phases), figsize=(6.3, 2.5), dpi=150, layout="constrained", squeeze=False,
        )
        for axis, (key, label) in zip(axes[0], phases.items()):
            values = encoding.dropna(subset=[key]).pivot(index="role", columns="model", values=key)
            values.plot.bar(
                ax=axis, color=[colors.get(model, "#537b96") for model in values.columns],
                rot=0, ylabel="CUDA ms/batch", xlabel="", width=0.55, legend=False,
            )
            axis.set_title(label, fontsize=9)
            axis.tick_params(labelsize=8)
            axis.set_axisbelow(True)
            axis.grid(axis="y", color="0.8", linestyle=":", linewidth=0.6)
        handles = {}
        for axis in axes[0]:
            artists, labels = axis.get_legend_handles_labels()
            handles.update(zip(labels, artists))
        figure.legend(
            handles.values(), handles.keys(), loc="outside upper center",
            ncol=len(handles), frameon=False, fontsize=8,
        )
        plt.show()
else:
    print("No encoding result is available at ENCODING_RESULT_PATH.")

## Duplicate control · matched corpus size

Job 29197143 compares a corpus of distinct texts with a quarter-sized sample
repeated to the same number of rows. Both corpora are shuffled. A negative
change means the repeated version was faster. The CUDA phase breakdown comes
from separate instrumented rounds.


In [4]:
unique = read_result(CONTROL_RESULT_DIR / "retrieval_cache_unique.json")
repeated = read_result(CONTROL_RESULT_DIR / "retrieval_cache_repeated.json")

if unique is not None and repeated is not None:
    unique_data = pd.json_normalize(unique["measurements"])
    repeated_data = pd.json_normalize(repeated["measurements"])
    paired = unique_data.merge(
        repeated_data, on=["method", "k"], suffixes=("_unique", "_repeated"),
        validate="one_to_one",
    )
    first = paired.iloc[0]
    display(Markdown(
        f"**{first['documents_unique']:,} rows per corpus** · "
        f"{unique['distinct_document_texts']:,} vs. {repeated['distinct_document_texts']:,} "
        f"distinct texts · {first['query_batch_size_unique']} queries per batch"
    ))
    paired["method_name"] = paired["method"].replace(
        {"qwen_dense": "Qwen3 (full)", "mrl": "MRL", "csrv2": "CSRv2"}
    )
    paired["width"] = paired["k"].astype("Int64").astype("string").fillna("Full")
    for metric, title in (
        ("ms_per_batch_mean", "Total search · ms/batch"),
        ("cuda_search_phases.scoring_ms_per_batch_mean", "CUDA scoring · ms/batch"),
        ("cuda_search_phases.selection_and_other_ms_per_batch_mean", "CUDA selection and other work · ms/batch"),
    ):
        baseline, trial = f"{metric}_unique", f"{metric}_repeated"
        if {baseline, trial}.issubset(paired.columns):
            paired["change_pct"] = 100 * (paired[trial] - paired[baseline]) / paired[baseline]
            show_table(title, paired, {
                "method_name": "Method", "width": "Width",
                baseline: "Unique", trial: "Repeated", "change_pct": "Change (%)",
            })


**240,746 rows per corpus** · 240,746 vs. 60,187 distinct texts · 512 queries per batch

**Total search · ms/batch**

,Method,Width,Unique,Repeated,Change (%)
0,Qwen3 (full),Full,44.042,44.024,-0.040
1,MRL,2,9.459,9.461,0.030
2,CSRv2,2,12.749,12.750,0.005
3,MRL,64,10.063,10.067,0.039
4,CSRv2,64,87.709,87.703,-0.006


**CUDA scoring · ms/batch**

,Method,Width,Unique,Repeated,Change (%)
0,Qwen3 (full),Full,35.108,35.069,-0.111
1,MRL,2,0.479,0.482,0.759
2,CSRv2,2,3.764,3.767,0.068
3,MRL,64,1.111,1.114,0.272
4,CSRv2,64,78.741,78.731,-0.013


**CUDA selection and other work · ms/batch**

,Method,Width,Unique,Repeated,Change (%)
0,Qwen3 (full),Full,9.027,9.034,0.085
1,MRL,2,9.080,9.068,-0.131
2,CSRv2,2,9.077,9.063,-0.147
3,MRL,64,9.036,9.031,-0.066
4,CSRv2,64,9.059,9.049,-0.111
